## Automated Evaluation Workflow with Langgraph and LLM

- AI evaluation workflow with Langgraph and openai.
- evaluate multiple criteria in parallel for faster , modular grading

Build an evaluation system that evaluates essays across key criteria - language quality, analysis depth and clarity - using parallel nodes that generate structure feedback and response. A final merge node computes simple and weighted averages.


In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("AZURE_OPENAI_KEY")
api_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
api_version = os.getenv("AZURE_OPENAI_VERSION")


In [2]:
default_temp = 0.2
prompt_version = "v1.3"

In [3]:
from langchain_openai import AzureChatOpenAI

llm = AzureChatOpenAI(
    azure_endpoint= api_endpoint,
    azure_deployment="gpt-4o-mini",
    openai_api_key=api_key,
    openai_api_version=api_version,
    temperature=default_temp)

In [5]:
from pydantic import BaseModel, Field
class EvaluationSchema(BaseModel):
    score: int = Field(description="The score for the answer, on a scale of 1 to 10.")
    feedback: str = Field(description="Detailed feedback for the criterion.")

In [6]:
structured_llm = llm.with_structured_output(EvaluationSchema)

In [7]:
from typing import Optional, TypedDict, Dict, List, Optional
class EvalState(TypedDict):
    essay: str
    feedback: Dict[str, str]
    scores: Dict[str, int]
    score_list : List[int]
    overall_feedback : Optional[str]
    simple_average: Optional[float]
    weighted_average: Optional[float]


In [9]:
RUBRIC = [
    {"key":"language",
     "prompt":"Evaluate the language used in the essay, including grammar, vocabulary, tone, structure and coherence. Provide a score from 1 to 10 and detailed feedback.\n\nEssay: {essay}",
     "weight":0.4},
    {"key":"analysis",
     "prompt":"Evaluate the DEPTH of analysis in the essay, including the quality of arguments, use of evidence, and critical thinking. Provide a score from 1 to 10 and detailed feedback.\n\nEssay: {essay}",
     "weight":0.33},
    {"key":"clarity",
     "prompt":"Evaluate the clarity of the essay, including the organization, logical flow, and overall readability. Provide a score from 1 to 10 and detailed feedback.\n\nEssay: {essay}",
     "weight":0.33}
]